# Offline Models Recipe

Use local model files from this repo. No Hugging Face connection is needed during class.

## 1. Switch Hugging Face to offline mode

In [ ]:
import os

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Offline mode:", os.environ["HF_HUB_OFFLINE"])

## 2. Use local folders, not Hub model IDs

In [ ]:
from pathlib import Path

ROOT = Path.cwd()
BGE_DIR = ROOT / "models" / "bge-small-en-v1.5-onnx"
BGE_ONNX = BGE_DIR / "onnx" / "model_quantized.onnx"
SMOLLM_DIR = ROOT / "models" / "smollm2-135m-instruct"

print(BGE_DIR)
print(SMOLLM_DIR)

## 3. Embeddings with BGE

`transformers` loads the tokenizer. ONNX Runtime loads the quantized model weights. The helper function `embed(texts)` returns one vector per input text.

In [ ]:
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BGE_DIR, local_files_only=True)
session = ort.InferenceSession(str(BGE_ONNX), providers=["CPUExecutionProvider"])

texts = [
    "The workshop runs fully offline.",
    "Students can clone the repo before class.",
    "Bananas are yellow fruit.",
]

def embed(texts):
    tokens = tokenizer(texts, padding=True, truncation=True, return_tensors="np")
    inputs = {x.name: tokens[x.name] for x in session.get_inputs()}

    # BGE stores the sentence meaning in the first token vector.
    token_vectors = session.run(None, inputs)[0]
    sentence_vectors = token_vectors[:, 0]

    # Normalize so a dot product becomes cosine similarity.
    return sentence_vectors / np.linalg.norm(sentence_vectors, axis=1, keepdims=True)


sentence_vectors = embed(texts)

print("Embeddings:", sentence_vectors.shape)
print("Similarities:")
print(np.round(sentence_vectors @ sentence_vectors.T, 3))

## 4. Text generation with SmolLM2

`transformers` loads both the tokenizer and the sharded `safetensors` model weights.

In [ ]:
import torch
from transformers import AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(SMOLLM_DIR, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
    SMOLLM_DIR,
    local_files_only=True,
    dtype="auto",
).eval()

messages = [
    {"role": "user", "content": "In one sentence, say why local model files help in an offline workshop."}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        repetition_penalty=1.1,
    )

answer_tokens = outputs[0, inputs["input_ids"].shape[-1]:]
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)
print(answer.strip())

## Key pattern

Use a local folder path plus `local_files_only=True`. Do not pass a Hub repo ID like `HuggingFaceTB/SmolLM2-135M-Instruct` during offline class.